In [188]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [189]:

df=pd.read_excel("Sales_Data_100Rows.xlsx")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Order ID       100 non-null    str    
 1   Order Date     73 non-null     str    
 2   Region         86 non-null     str    
 3   Category       100 non-null    str    
 4   Sales          65 non-null     float64
 5   Profit         42 non-null     float64
 6   Quantity       67 non-null     object 
 7   Customer Name  83 non-null     str    
dtypes: float64(2), object(1), str(5)
memory usage: 6.4+ KB


In [190]:
df
# df.duplicated().sum()

,Order ID,Order Date,Region,Category,Sales,Profit,Quantity,Customer Name
0,ORD058,24-01-2024,EAST,Technology,-1515.0,NaN,two,Rakesh
1,ORD023,11-01-2024,NaN,Tech,NaN,NaN,4,Priya
2,ORD027,12/01/24,NaN,Technology,20087.0,NaN,3,Meena
3,ORD041,11/01/24,West,Office supplies,NaN,788.0,3,Kunal
4,ORD010,18-01-2024,NaN,Technology,NaN,5430.0,1,Kunal
...,...,...,...,...,...,...,...,...
95,ORD030,NaN,West,Office supplies,NaN,NaN,two,Kunal
96,ORD073,NaN,NORTH,Tech,NaN,3643.0,NaN,NaN
97,ORD048,19-01-2024,West,Office supplies,2483.0,NaN,1,Priya
98,ORD023,NaN,West,Tech,NaN,NaN,two,Riya


In [191]:
df.isnull().sum()

Order ID          0
Order Date       27
Region           14
Category          0
Sales            35
Profit           58
Quantity         33
Customer Name    17
dtype: int64

In [192]:
# df["Quantity"]=pd.to_numeric(df['Quantity'])
df['Quantity']=df['Quantity'].replace({"two":2})
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
df

,Order ID,Order Date,Region,Category,Sales,Profit,Quantity,Customer Name
0,ORD058,24-01-2024,EAST,Technology,-1515.0,NaN,2.0,Rakesh
1,ORD023,11-01-2024,NaN,Tech,NaN,NaN,4.0,Priya
2,ORD027,12/01/24,NaN,Technology,20087.0,NaN,3.0,Meena
3,ORD041,11/01/24,West,Office supplies,NaN,788.0,3.0,Kunal
4,ORD010,18-01-2024,NaN,Technology,NaN,5430.0,1.0,Kunal
...,...,...,...,...,...,...,...,...
95,ORD030,NaN,West,Office supplies,NaN,NaN,2.0,Kunal
96,ORD073,NaN,NORTH,Tech,NaN,3643.0,NaN,NaN
97,ORD048,19-01-2024,West,Office supplies,2483.0,NaN,1.0,Priya
98,ORD023,NaN,West,Tech,NaN,NaN,2.0,Riya


In [193]:
df['Profit'] = df['Profit'].fillna(df['Profit'].median())
df['Sales'] = df['Sales'].fillna(df['Sales'].median())
df['Quantity'] = df['Quantity'].fillna(df['Quantity'].median())
df['Region'] = df['Region'].str.lower()
df['Region'] = df['Region'].fillna(df['Region'].mode()[0])
df['Category'] = df['Category'].fillna(df['Category'].mode()[0])

df['Order Date'] = df['Order Date'].astype(str).str.strip()
df['Order Date'] = df['Order Date'].str.replace('/', '-', regex=False)
df['Order Date'] = df['Order Date'].str.replace(
    r'(\d{2}-\d{2})-(\d{2})$',
    r'\1-20\2',
    regex=True
)
df['Order Date'] = pd.to_datetime(
    df['Order Date'],
    dayfirst=True,
    errors='coerce'
)
df['Order Date'] = df['Order Date'].ffill()
df['Customer Name'] = df['Customer Name'].fillna('idk')
df = df.drop(columns=['Order ID'], errors='ignore')
df

,Order Date,Region,Category,Sales,Profit,Quantity,Customer Name
0,2024-01-24,east,Technology,-1515.0,3002.5,2.0,Rakesh
1,2024-01-11,north,Tech,2036.0,3002.5,4.0,Priya
2,2024-01-12,north,Technology,20087.0,3002.5,3.0,Meena
3,2024-01-11,west,Office supplies,2036.0,788.0,3.0,Kunal
4,2024-01-18,north,Technology,2036.0,5430.0,1.0,Kunal
...,...,...,...,...,...,...,...
95,2024-01-02,west,Office supplies,2036.0,3002.5,2.0,Kunal
96,2024-01-02,north,Tech,2036.0,3643.0,2.0,idk
97,2024-01-19,west,Office supplies,2483.0,3002.5,1.0,Priya
98,2024-01-19,west,Tech,2036.0,3002.5,2.0,Riya


In [195]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
model_df = df.copy()
if 'Order Date' in model_df.columns:
    model_df['Order Date'] = pd.to_datetime(model_df['Order Date'], errors='coerce')
    model_df['Order_Day'] = model_df['Order Date'].dt.day
    model_df['Order_Month'] = model_df['Order Date'].dt.month
    model_df['Order_Year'] = model_df['Order Date'].dt.year
    model_df = model_df.drop(columns=['Order Date'])
model_df = model_df.dropna(axis=1, how='all')
obj_cols = model_df.select_dtypes(include=['object', 'string']).columns
model_df = pd.get_dummies(model_df, columns=obj_cols, drop_first=True)
model_df['Target'] = (model_df['Profit'] > 0).astype(int)
X = model_df.drop('Target', axis=1)
y = model_df['Target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)
lr = LogisticRegression(max_iter=2000)
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)
print('=== Decision Tree ===')
print('Accuracy:', accuracy_score(y_test, y_pred_dt))
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred_dt))
print('\n=== Logistic Regression ===')
print('Accuracy:', accuracy_score(y_test, y_pred_lr))
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred_lr))

=== Decision Tree ===
Accuracy: 1.0
Confusion Matrix:
 [[ 2  0]
 [ 0 18]]

=== Logistic Regression ===
Accuracy: 0.95
Confusion Matrix:
 [[ 1  1]
 [ 0 18]]
